In [0]:
%run ./00_logging

DataFrame[]

In [0]:
from datetime import datetime
from pyspark.sql import functions as F

dbutils.widgets.text("batch_date", "2026-10-14")
dbutils.widgets.dropdown("load_type", "incremental", ["full", "incremental"])
BATCH_DATE = dbutils.widgets.get("batch_date")
LOAD_TYPE = dbutils.widgets.get("load_type")
print(f"Silver -> Gold | batch_date={BATCH_DATE} load_type={LOAD_TYPE}")

S = "workspace.silver"
G = "workspace.gold"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {G}")

def repo_key(c="repo_name"):  return F.xxhash64(F.col(c)).alias("repo_key")
def user_key(c): return F.when(F.col(c).isNotNull(), F.xxhash64(F.col(c)))
def date_key(c):              return F.date_format(F.col(c), "yyyyMMdd").cast("int")

def merge_into(df, target, keys, update=True):
    """MERGE df into target on keys. Returns (inserted, updated)."""
    if not spark.catalog.tableExists(target):
        df.limit(0).write.format("delta").saveAsTable(target)
    df.createOrReplaceTempView("g_src")
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    upd = "WHEN MATCHED THEN UPDATE SET *" if update else ""
    r = spark.sql(f"""MERGE WITH SCHEMA EVOLUTION INTO {target} t USING g_src s ON {cond}
                      {upd} WHEN NOT MATCHED THEN INSERT *""").collect()[0]
    return int(r["num_inserted_rows"]), int(r["num_updated_rows"])

def run_step(name, fn):
    start = datetime.now()
    param = f"{LOAD_TYPE}/{BATCH_DATE}/{name}"
    try:
        ins, upd = fn()
        write_log("silver_to_gold", name, param, LOAD_TYPE, start, "SUCCESS", ins, upd, None)
        print(f"{name}: inserted={ins} updated={upd}")
    except Exception as e:
        write_log("silver_to_gold", name, param, LOAD_TYPE, start, "FAILURE", 0, 0, str(e)[:500])
        print(f"{name}: FAILED -> {str(e)[:300]}")
        raise

Silver -> Gold | batch_date=2026-10-10 load_type=incremental


In [0]:
def build_dim_repo():
    repos = (spark.table(f"{S}.commits").select("repo_name")
             .union(spark.table(f"{S}.issues").select("repo_name"))
             .union(spark.table(f"{S}.pulls").select("repo_name")).distinct())
    df = repos.select(repo_key(), "repo_name",
                      F.element_at(F.split("repo_name", "/"), 1).alias("owner"),
                      F.element_at(F.split("repo_name", "/"), -1).alias("repo"))
    return merge_into(df, f"{G}.dim_repo", ["repo_key"], update=False)

def build_dim_user():
    users = (spark.table(f"{S}.commits").select(F.col("author_login").alias("login"))
             .union(spark.table(f"{S}.issues").select(F.col("user_login").alias("login")))
             .union(spark.table(f"{S}.pulls").select(F.col("user_login").alias("login")))
             .where(F.col("login").isNotNull()).distinct())
    df = users.select(user_key("login").alias("user_key"), "login")
    return merge_into(df, f"{G}.dim_user", ["user_key"], update=False)

def build_dim_date():
    df = (spark.sql("SELECT explode(sequence(DATE'2025-01-01', DATE'2027-12-31', INTERVAL 1 DAY)) AS d")
          .select(date_key("d").alias("date_key"), F.col("d").alias("date"),
                  F.year("d").alias("year"), F.quarter("d").alias("quarter"),
                  F.month("d").alias("month"), F.date_format("d", "MMM").alias("month_name"),
                  F.weekofyear("d").alias("week_of_year"),
                  F.dayofweek("d").alias("day_of_week"),
                  F.date_format("d", "EEEE").alias("day_name"),
                  F.dayofweek("d").isin(1, 7).alias("is_weekend")))
    return merge_into(df, f"{G}.dim_date", ["date_key"], update=False)

run_step("dim_repo", build_dim_repo)
run_step("dim_user", build_dim_user)
run_step("dim_date", build_dim_date)

dim_repo: inserted=0 updated=0
dim_user: inserted=0 updated=0
dim_date: inserted=0 updated=0


In [0]:
def batch(df):
    return df.where((F.col("batch_date") == BATCH_DATE) & (F.col("load_type") == LOAD_TYPE))

def build_fact_commits():
    df = (batch(spark.table(f"{S}.commits"))
          .select(repo_key(), "sha",
                  date_key("commit_date").alias("commit_date_key"),
                  user_key("author_login").alias("author_key"),
                  "author_email_hash", "committer_email_hash",
                  F.length("commit_message").alias("message_length"),
                  F.col("commit_date").alias("commit_ts"),
                  "batch_date", F.current_timestamp().alias("load_timestamp"))
          .dropDuplicates(["repo_key", "sha"]))
    return merge_into(df, f"{G}.fact_commits", ["repo_key", "sha"])

def build_fact_issues():
    df = (batch(spark.table(f"{S}.issues")).where(~F.col("is_pr"))
          .select(repo_key(), F.col("id").alias("issue_id"), "number", "state",
                  date_key("created_at_ts").alias("created_date_key"),
                  date_key("closed_at_ts").alias("closed_date_key"),
                  user_key("user_login").alias("user_key"),
                  (F.col("state") == "closed").alias("is_closed"),
                  "label_count", "comments", "time_to_close_hours", "updated_at_ts",
                  "batch_date", F.current_timestamp().alias("load_timestamp"))
          .dropDuplicates(["repo_key", "issue_id"]))
    return merge_into(df, f"{G}.fact_issues", ["repo_key", "issue_id"])

def build_fact_pulls():
    df = (batch(spark.table(f"{S}.pulls"))
          .select(repo_key(), F.col("id").alias("pr_id"), "number", "state",
                  date_key("created_at_ts").alias("created_date_key"),
                  date_key("closed_at_ts").alias("closed_date_key"),
                  date_key("merged_at_ts").alias("merged_date_key"),
                  user_key("user_login").alias("user_key"),
                  F.coalesce(F.col("merged"), F.lit(False)).alias("is_merged"),
                  "comments", "review_comments", "commits", "additions", "deletions",
                  "changed_files", "time_to_merge_hours", "updated_at_ts",
                  "batch_date", F.current_timestamp().alias("load_timestamp"))
          .dropDuplicates(["repo_key", "pr_id"]))
    return merge_into(df, f"{G}.fact_pulls", ["repo_key", "pr_id"])

run_step("fact_commits", build_fact_commits)
run_step("fact_issues", build_fact_issues)
run_step("fact_pulls", build_fact_pulls)

fact_commits: inserted=0 updated=87
fact_issues: inserted=0 updated=38
fact_pulls: inserted=0 updated=198


In [0]:
def build_agg_repo_daily():
    c = (spark.table(f"{G}.fact_commits").groupBy("repo_key", F.col("commit_date_key").alias("date_key"))
         .agg(F.count("*").alias("commits"), F.countDistinct("author_key").alias("active_authors")))
    io = (spark.table(f"{G}.fact_issues").groupBy("repo_key", F.col("created_date_key").alias("date_key"))
          .agg(F.count("*").alias("issues_opened")))
    ic = (spark.table(f"{G}.fact_issues").where(F.col("closed_date_key").isNotNull())
          .groupBy("repo_key", F.col("closed_date_key").alias("date_key"))
          .agg(F.count("*").alias("issues_closed"), F.avg("time_to_close_hours").alias("avg_time_to_close_hours")))
    po = (spark.table(f"{G}.fact_pulls").groupBy("repo_key", F.col("created_date_key").alias("date_key"))
          .agg(F.count("*").alias("prs_opened")))
    pm = (spark.table(f"{G}.fact_pulls").where(F.col("merged_date_key").isNotNull())
          .groupBy("repo_key", F.col("merged_date_key").alias("date_key"))
          .agg(F.count("*").alias("prs_merged"), F.avg("time_to_merge_hours").alias("avg_time_to_merge_hours")))
    keys = ["repo_key", "date_key"]
    df = c
    for other in (io, ic, po, pm):
        df = df.join(other, keys, "full_outer")
    cnt = ["commits", "active_authors", "issues_opened", "issues_closed", "prs_opened", "prs_merged"]
    df = df.fillna(0, subset=cnt).withColumn("load_timestamp", F.current_timestamp())
    old = spark.table(f"{G}.agg_repo_daily").count() if spark.catalog.tableExists(f"{G}.agg_repo_daily") else 0
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{G}.agg_repo_daily")
    new = spark.table(f"{G}.agg_repo_daily").count()
    return max(new - old, 0), min(old, new)

run_step("agg_repo_daily", build_agg_repo_daily)

agg_repo_daily: inserted=0 updated=2793


In [0]:
dd = spark.table(f"{G}.dim_date").select(F.col("date_key").alias("dd_key"), "year", "month")
dr = spark.table(f"{G}.dim_repo").select("repo_key", "repo_name")
du = spark.table(f"{G}.dim_user").select("user_key", "login")
fc = spark.table(f"{G}.fact_commits")
fi = spark.table(f"{G}.fact_issues")
fp = spark.table(f"{G}.fact_pulls")

def with_ym(df, kcol):
    return df.join(F.broadcast(dd), df[kcol] == dd.dd_key, "left").drop("dd_key")

def overwrite(df, name):
    t = f"{G}.{name}"
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(t)
    return spark.table(t).count(), 0

def build_gold_contributor_activity():
    k = ["repo_key", "user_key", "year", "month"]
    c = (with_ym(fc, "commit_date_key").groupBy("repo_key", F.col("author_key").alias("user_key"), "year", "month")
         .agg(F.count("*").alias("commits")))
    po = (with_ym(fp, "created_date_key").groupBy(*k).agg(F.count("*").alias("prs_opened")))
    pm = (with_ym(fp.where(F.col("merged_date_key").isNotNull()), "merged_date_key")
          .groupBy(*k).agg(F.count("*").alias("prs_merged")))
    df = (c.join(po, k, "full_outer").join(pm, k, "full_outer")
          .where(F.col("user_key").isNotNull())
          .fillna(0, subset=["commits", "prs_opened", "prs_merged"])
          .join(du, "user_key", "left").join(dr, "repo_key", "left"))
    return overwrite(df, "gold_contributor_activity")

def build_gold_repo_momentum():
    df = (with_ym(spark.table(f"{G}.agg_repo_daily"), "date_key")
          .groupBy("repo_key", "year", "month")
          .agg(F.sum("commits").alias("commits"), F.sum("issues_opened").alias("issues_opened"),
               F.sum("issues_closed").alias("issues_closed"), F.sum("prs_opened").alias("prs_opened"),
               F.sum("prs_merged").alias("prs_merged"))
          .join(dr, "repo_key", "left"))
    return overwrite(df, "gold_repo_momentum")

def build_gold_issue_pr_turnaround():
    k = ["repo_key", "year", "month"]
    ic = (with_ym(fi.where(F.col("closed_date_key").isNotNull()), "closed_date_key").groupBy(*k)
          .agg(F.count("*").alias("issues_closed"),
               F.avg("time_to_close_hours").alias("avg_hours_to_close"),
               F.percentile_approx("time_to_close_hours", 0.5).alias("median_hours_to_close")))
    pm = (with_ym(fp.where(F.col("merged_date_key").isNotNull()), "merged_date_key").groupBy(*k)
          .agg(F.count("*").alias("prs_merged"),
               F.avg("time_to_merge_hours").alias("avg_hours_to_merge"),
               F.percentile_approx("time_to_merge_hours", 0.5).alias("median_hours_to_merge")))
    df = ic.join(pm, k, "full_outer").join(dr, "repo_key", "left")
    return overwrite(df, "gold_issue_pr_turnaround")

def build_gold_repo_summary():
    a = fc.groupBy("repo_key").agg(F.count("*").alias("total_commits"),
                                   F.countDistinct("author_key").alias("total_contributors"))
    b = fi.groupBy("repo_key").agg(F.count("*").alias("total_issues"),
                                   F.sum(F.when(F.col("is_closed"), 1).otherwise(0)).alias("closed_issues"))
    c = fp.groupBy("repo_key").agg(F.count("*").alias("total_prs"),
                                   F.sum(F.when(F.col("is_merged"), 1).otherwise(0)).alias("merged_prs"),
                                   F.avg("time_to_merge_hours").alias("avg_hours_to_merge"))
    df = (a.join(b, "repo_key", "full_outer").join(c, "repo_key", "full_outer")
          .withColumn("open_issues", F.col("total_issues") - F.col("closed_issues"))
          .withColumn("open_issue_ratio", F.col("open_issues") / F.col("total_issues"))
          .withColumn("pr_merge_rate", F.col("merged_prs") / F.col("total_prs"))
          .join(dr, "repo_key", "left"))
    return overwrite(df, "gold_repo_summary")

run_step("gold_contributor_activity", build_gold_contributor_activity)
run_step("gold_repo_momentum", build_gold_repo_momentum)
run_step("gold_issue_pr_turnaround", build_gold_issue_pr_turnaround)
run_step("gold_repo_summary", build_gold_repo_summary)

gold_contributor_activity: inserted=2878 updated=0
gold_repo_momentum: inserted=85 updated=0
gold_issue_pr_turnaround: inserted=84 updated=0
gold_repo_summary: inserted=4 updated=0


In [0]:
CLUSTER = {
    "fact_commits": "repo_key, commit_date_key",
    "fact_issues": "repo_key, created_date_key",
    "fact_pulls": "repo_key, created_date_key",
    "agg_repo_daily": "repo_key, date_key",
}
for t, cols in CLUSTER.items():
    try:
        spark.sql(f"ALTER TABLE {G}.{t} CLUSTER BY ({cols})")
        spark.sql(f"OPTIMIZE {G}.{t}")
        print(f"optimized {t} (clustered by {cols})")
    except Exception as e:
        print(f"optimize skipped for {t}: {str(e)[:200]}")

optimized fact_commits (clustered by repo_key, commit_date_key)
optimized fact_issues (clustered by repo_key, created_date_key)
optimized fact_pulls (clustered by repo_key, created_date_key)
optimized agg_repo_daily (clustered by repo_key, date_key)


In [0]:
%sql
SELECT layer, entity, load_type, status, rows_inserted, rows_updated,
       round(unix_timestamp(end_time) - unix_timestamp(start_time), 1) AS seconds
FROM workspace.ops.pipeline_execution_logs
WHERE layer = 'silver_to_gold' ORDER BY start_time DESC LIMIT 20;

layer,entity,load_type,status,rows_inserted,rows_updated,seconds
silver_to_gold,gold_repo_summary,incremental,SUCCESS,4,0,4
silver_to_gold,gold_issue_pr_turnaround,incremental,SUCCESS,84,0,3
silver_to_gold,gold_repo_momentum,incremental,SUCCESS,85,0,3
silver_to_gold,gold_contributor_activity,incremental,SUCCESS,2878,0,4
silver_to_gold,agg_repo_daily,incremental,SUCCESS,0,2793,4
silver_to_gold,fact_pulls,incremental,SUCCESS,0,198,7
silver_to_gold,fact_issues,incremental,SUCCESS,0,38,7
silver_to_gold,fact_commits,incremental,SUCCESS,0,87,7
silver_to_gold,dim_date,incremental,SUCCESS,0,0,2
silver_to_gold,dim_user,incremental,SUCCESS,0,0,3


In [0]:
%sql
SELECT 'fact_commits' t, count(*) n, count(DISTINCT repo_key, sha) d FROM workspace.gold.fact_commits
UNION ALL SELECT 'fact_issues', count(*), count(DISTINCT repo_key, issue_id) FROM workspace.gold.fact_issues
UNION ALL SELECT 'fact_pulls', count(*), count(DISTINCT repo_key, pr_id) FROM workspace.gold.fact_pulls;

t,n,d
fact_commits,3079,3079
fact_issues,1630,1630
fact_pulls,5819,5819
